# SLUG — Shot Review

Thin notebook: all the real logic (shot organization, decoding, pulse characterization,
plotting) lives in `shot_log.py`, `helpers/`, and `shot_characterization/` one level up.
This notebook just calls into those.

Run the cells top to bottom, then use **"Step through shots"** at the bottom to page
through shots one at a time.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # so we can import shot_log / helpers / shot_characterization

import pandas as pd
import matplotlib.pyplot as plt

import shot_log as sl
from helpers import load_channel, plot_shot
from shot_characterization import characterize

print(f"{len(sl.ALL_SHOTS)} total shots: {len(sl.GOOD_SHOTS)} good, {len(sl.BAD_SHOTS)} bad/blocked/missing")
print(f"DATA_DIR = {sl.DATA_DIR}")


## Inspect one shot

In [ ]:
def show_shot(shot, zoom_pad_ns=50):
    info = sl.SHOT_LOG.get(shot)
    entries = sl.FILE_MAP.get(shot, [])
    print(f"{'='*70}\nSHOT {shot}\n{'='*70}")
    if info:
        thick = info["thickness_um"]
        thick_s = f"{thick} um" if thick is not None else "?"
        status = "GOOD SHOT" if info["good"] else "NOT GOOD"
        print(f"target: {info['target']}  thickness: {thick_s}  [{status}]")
        if info["note"]:
            print(f"log note: {info['note']}")
    if not entries:
        print("\n*** no data file for this shot ***")
        return

    results = plot_shot(entries, load_channel, characterize, sl.DATA_DIR, zoom_pad_ns=zoom_pad_ns)
    for (kind, path, ch), res in zip(entries, results):
        tag = "MAIN" if (kind, path, ch) == sl.main_channel_entry(shot) else ""
        print(f"\n--- {ch} ({kind}: {path}) {tag} ---")
        print(f"  category: {res['category']}   snr={res['snr']:.1f}   "
              f"clipped={res['clipped']}   peak_amp={res['peak_amp_V']:.4g} V   "
              f"n_points={res['n_points']:,}")
        xr, pr = res["xray_candidate"], res["proton_candidate"]
        if xr:
            print(f"  x-ray candidate:   t={xr['peak_time_ns']:.3f} ns   V={xr['peak_val_V']:.5f}")
        if pr:
            sep = pr["peak_time_ns"] - xr["peak_time_ns"]
            print(f"  proton candidate:  t={pr['peak_time_ns']:.3f} ns   V={pr['peak_val_V']:.5f}   (sep={sep:.3f} ns)")
    plt.show()

show_shot(9)


## Step through shots one by one

Run the cell below repeatedly (Ctrl+Enter) to advance through `SHOT_SET` in order.


In [ ]:
SHOT_SET = sl.ALL_SHOTS  # change to sl.GOOD_SHOTS to only step through the good ones
_i = {"n": -1}

def next_shot():
    _i["n"] = (_i["n"] + 1) % len(SHOT_SET)
    print(f"[{_i['n']+1}/{len(SHOT_SET)}]")
    show_shot(SHOT_SET[_i["n"]])


In [ ]:
next_shot()  # run again to advance


## Summary table (every shot at once)

In [ ]:
rows = []
for shot in sl.ALL_SHOTS:
    info = sl.SHOT_LOG[shot]
    entry = sl.main_channel_entry(shot)
    base = dict(shot=shot, target=info["target"], thickness_um=info["thickness_um"],
                good=info["good"], log_note=info["note"])
    if entry is None:
        rows.append(dict(base, channel=None, category="NO DATA FILE",
                          snr=None, xray_t_ns=None, proton_t_ns=None, sep_ns=None))
        continue
    kind, path, ch = entry
    t, v, clipped, ydisp = load_channel(sl.DATA_DIR, kind, path, ch)
    res = characterize(t, v, clipped)
    xr, pr = res["xray_candidate"], res["proton_candidate"]
    rows.append(dict(
        base, channel=ch, category=res["category"], snr=round(res["snr"], 1),
        xray_t_ns=round(xr["peak_time_ns"], 2) if xr else None,
        proton_t_ns=round(pr["peak_time_ns"], 2) if pr else None,
        sep_ns=round(pr["peak_time_ns"] - xr["peak_time_ns"], 2) if (xr and pr) else None,
    ))

summary_df = pd.DataFrame(rows).set_index("shot")
summary_df


In [ ]:
# summary_df.to_csv('../results/summary.csv')
